# Phone sensor analysis (accelerometer + gyroscope)

Goal: check whether the Android sensor values are **useful and trustworthy**.

Steps, from simple to more advanced:
1. Load and plot raw values
2. Sampling rate check (timing quality)
3. Accelerometer magnitude vs. gravity (calibration check)
4. Noise and bias while the phone lies still
5. Frequency spectrum
6. Tilt from accelerometer vs. integrated gyroscope (do both sensors agree?)
7. Gyro drift when integrating

## 1. Load and plot raw values

In [25]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# thinner default lines for all plots
thin = go.layout.Template(pio.templates["plotly"])
thin.data.scatter[0].line.width = 0.6
pio.templates["thin"] = thin
pio.templates.default = "thin"

G = 9.81  # gravity in m/s^2
NAME = "20261020_131029"


def load(sensor: str):
    """Read one CSV: returns time [s] and an (n, 3) array with x, y, z."""
    data = np.loadtxt(f"data/{sensor}_{NAME}.csv", delimiter=",", skiprows=1)
    return data[:, 0] / 1e9, data[:, 1:]  # ns -> s


t, acc = load("accelerometer")  # m/s^2
t_gyr, gyr_raw = load("gyroscope")  # rad/s

# shared time axis starting at 0
t0 = t[0]
t, t_gyr = t - t0, t_gyr - t0

# sensors have separate timestamps -> resample gyro onto accelerometer time
gyr = np.column_stack([np.interp(t, t_gyr, gyr_raw[:, i]) for i in range(3)])

print(f"samples: {len(t)}, duration: {t[-1]:.1f} s")

samples: 3750, duration: 30.0 s


In [26]:
# Raw values of all 6 channels over time.
# One row per axis, so each gets its own y-scale and small changes stay visible.
COLORS = {"x": "#1f77b4", "y": "#ff7f0e", "z": "#2ca02c"}

titles = [f"acc {axis} [m/s²]" for axis in "xyz"] + [f"gyro {axis} [rad/s]" for axis in "xyz"]
fig = make_subplots(rows=6, cols=1, shared_xaxes=True, vertical_spacing=0.03, subplot_titles=titles)
for i, axis in enumerate("xyz"):
    fig.add_trace(go.Scatter(x=t, y=acc[:, i], name=f"acc {axis}", line_color=COLORS[axis]), row=i + 1, col=1)  # rows 1-3
    fig.add_trace(go.Scatter(x=t_gyr, y=gyr_raw[:, i], name=f"gyro {axis}", line_color=COLORS[axis]), row=i + 4, col=1)  # rows 4-6

fig.update_xaxes(title_text="time [s]", row=6, col=1)
fig.update_layout(height=1400, hovermode="x unified", showlegend=False, title="Raw sensor values")
fig.show()

### Interpretation (recording 20261020_131029, hand on chest)
- **Baseline:** acc z ≈ 9.8 m/s² is gravity → phone lies almost flat. acc x ≈ 1.4 and y ≈ −0.5 → tilted by about 8°. Gyro centered at 0 → no rotation.
- **0–1.5 s:** large swings on all axes = settling the hand after pressing start. Ignore this part.
- **~14 s:** one sharp spike on all axes at once = a single movement (twitch, swallow, shifting the hand).
- **Breathing:** slow waves in acc x/y, period ~3.5 s. FFT peak at **0.29 Hz ≈ 17 breaths/min** on all three axes.
- **Heartbeat (likely):** small bursts in acc z and gyro, about one per second. acc x peaks at **1.14 Hz ≈ 68 bpm**. The bursts in acc z come at ~2× that rate, which fits the two heart sounds per beat ("lub-dub"). Not yet confirmed; compare with a pulse measured at the wrist.

**Conclusion:** the sensors resolve breathing clearly and very probably the heartbeat → sensitive and plausible data.

## 2. Sampling rate check
A trustworthy sensor delivers samples at a steady rate. Look for jitter and gaps.

In [27]:
# Time between consecutive samples: should be constant (125 Hz -> 8 ms).
dt = np.diff(t) * 1000  # ms

print(f"mean rate: {1000 / dt.mean():.1f} Hz")
print(f"dt: median {np.median(dt):.3f} ms, min {dt.min():.3f} ms, max {dt.max():.3f} ms, std {dt.std():.4f} ms")
print(f"gaps > 2x median: {(dt > 2 * np.median(dt)).sum()}")
print(f"gyro starts {t_gyr[0] * 1000:+.2f} ms vs. accelerometer")

# Histogram: one narrow bar = steady timing, wide spread = jitter
fig = go.Figure(go.Histogram(x=dt, nbinsx=50))
fig.update_layout(height=350, xaxis_title="time between samples [ms]", yaxis_title="count", title="Sample interval distribution")
fig.show()

mean rate: 125.0 Hz
dt: median 7.999 ms, min 7.988 ms, max 8.013 ms, std 0.0033 ms
gaps > 2x median: 0
gyro starts -4.00 ms vs. accelerometer


## 3. Accelerometer magnitude vs. gravity
When the phone is not accelerating, |a| must be ≈ 9.81 m/s², no matter how it is oriented.
Deviations at rest = calibration error (offset/scale).

In [28]:
# Total length of the vectors, independent of how the phone is oriented.
# |a| should stay at g (red line) when the phone is at rest, |ω| near 0.
acc_mag = np.linalg.norm(acc, axis=1)
gyr_mag = np.linalg.norm(gyr, axis=1)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06)
fig.add_trace(go.Scatter(x=t, y=acc_mag, name="|a|"), row=1, col=1)
fig.add_hline(y=G, line_dash="dash", line_color="red", annotation_text="g = 9.81", row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=gyr_mag, name="|ω|"), row=2, col=1)

fig.update_yaxes(title_text="|a| [m/s²]", row=1, col=1)
fig.update_yaxes(title_text="|ω| [rad/s]", row=2, col=1)
fig.update_xaxes(title_text="time [s]", row=2, col=1)
fig.update_layout(height=500, hovermode="x unified", title="Magnitudes")
fig.show()

## 4. Noise and bias while still
Find the segments where the phone lies still (low gyro rate over a short window).
There, the gyro should read 0 (anything else = **bias**) and the spread is the **noise**.

In [29]:
WINDOW = 50  # samples (~0.4 s)
STILL_THRESHOLD = 0.05  # rad/s


def rolling_mean(values, n):
    return np.convolve(values, np.ones(n) / n, mode="same")


# "still" = average rotation over the window is below the threshold
still = rolling_mean(gyr_mag, WINDOW) < STILL_THRESHOLD
print(f"still samples: {still.sum()} ({still.mean():.0%} of recording)")

# Green areas = segments used as "at rest" in the next cells
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=t, y=np.where(still, gyr_mag.max(), 0), name="still", fill="tozeroy", line_width=0, fillcolor="rgba(44, 160, 44, 0.2)", hoverinfo="skip")
)
fig.add_trace(go.Scatter(x=t, y=gyr_mag, name="|ω|"))
fig.update_layout(height=350, xaxis_title="time [s]", yaxis_title="|ω| [rad/s]", title="Still segments")
fig.show()

still samples: 3348 (89% of recording)


In [30]:
# Only the samples at rest
acc_still = acc[still]
gyr_still = gyr[still]

# mean = offset (gyro should be 0), std = noise level
print("Accelerometer at rest:")
print(f"  mean [m/s²]: {acc_still.mean(axis=0).round(4)}")
print(f"  noise std  : {acc_still.std(axis=0).round(4)}")
print(f"  |a| mean   : {np.linalg.norm(acc_still, axis=1).mean():.4f} (error vs g: {np.linalg.norm(acc_still, axis=1).mean() - G:+.4f})")
print("Gyroscope at rest:")
print(f"  bias [rad/s]: {gyr_still.mean(axis=0).round(5)}")
print(f"  bias [deg/s]: {np.degrees(gyr_still.mean(axis=0)).round(3)}")
print(f"  noise std   : {gyr_still.std(axis=0).round(5)}")

Accelerometer at rest:
  mean [m/s²]: [ 1.3516 -0.4453  9.7159]
  noise std  : [0.0804 0.1667 0.0897]
  |a| mean   : 9.8213 (error vs g: +0.0113)
Gyroscope at rest:
  bias [rad/s]: [ 0.00071 -0.00146  0.00054]
  bias [deg/s]: [ 0.041 -0.084  0.031]
  noise std   : [0.02628 0.02405 0.00702]


In [31]:
# Distribution of the values at rest, per axis.
# Narrow bell shape = pure sensor noise, wide or multiple humps = real motion (breathing, heartbeat, hand).
titles = [f"acc {axis} at rest" for axis in "xyz"] + [f"gyro {axis} at rest" for axis in "xyz"]
fig = make_subplots(rows=2, cols=3, subplot_titles=titles)
for i, axis in enumerate("xyz"):
    fig.add_trace(go.Histogram(x=acc_still[:, i], nbinsx=40, marker_color=COLORS[axis]), row=1, col=i + 1)
    fig.add_trace(go.Histogram(x=gyr_still[:, i], nbinsx=40, marker_color=COLORS[axis]), row=2, col=i + 1)
fig.update_layout(height=550, showlegend=False, title="Noise distribution at rest")
fig.show()

## 5. Frequency spectrum
Shows where the signal energy is: slow hand motion (< 5 Hz), tremor, vibrations or noise.
A flat floor at higher frequencies is typical white sensor noise.

In [32]:
# FFT: splits each signal into frequencies. Peaks = repeating motions.
# Expect breathing around 0.2-0.4 Hz and heartbeat around 1-2 Hz.
rate = 1 / np.median(np.diff(t))
freqs = np.fft.rfftfreq(len(t), 1 / rate)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Accelerometer spectrum", "Gyroscope spectrum"))
for i, axis in enumerate("xyz"):
    # subtract the mean, otherwise the constant part (gravity) dominates at 0 Hz
    fig.add_trace(go.Scatter(x=freqs, y=np.abs(np.fft.rfft(acc[:, i] - acc[:, i].mean())), name=axis, legendgroup=axis, line_color=COLORS[axis]), row=1, col=1)
    fig.add_trace(
        go.Scatter(x=freqs, y=np.abs(np.fft.rfft(gyr[:, i] - gyr[:, i].mean())), name=axis, legendgroup=axis, line_color=COLORS[axis], showlegend=False),
        row=1,
        col=2,
    )
fig.update_yaxes(type="log")
fig.update_xaxes(title_text="frequency [Hz]")
fig.update_layout(height=450)
fig.show()

## 6. Do both sensors agree? Tilt comparison
Two independent ways to get the phone's tilt:
- **Accelerometer**: direction of gravity → roll/pitch (noisy during motion, but no drift)
- **Gyroscope**: integrate angular rate over time (smooth, but drifts)

If both curves match, both sensors are consistent → trustworthy.
(Simple approximation, works best for moderate tilts.)

In [33]:
# Tilt from the accelerometer: angle of the gravity vector
ax_, ay_, az_ = acc.T
roll_acc = np.degrees(np.arctan2(ay_, az_))
pitch_acc = np.degrees(np.arctan2(-ax_, np.hypot(ay_, az_)))

# Tilt from the gyroscope: sum up angular rate * time step, starting at the accelerometer angle
dt_s = np.diff(t, prepend=t[0])
roll_gyr = roll_acc[0] + np.degrees(np.cumsum(gyr[:, 0] * dt_s))
pitch_gyr = pitch_acc[0] + np.degrees(np.cumsum(gyr[:, 1] * dt_s))

# Both curves should follow each other; a slowly growing gap = gyro drift
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06)
for row, (from_acc, from_gyr) in enumerate([(roll_acc, roll_gyr), (pitch_acc, pitch_gyr)], start=1):
    fig.add_trace(go.Scatter(x=t, y=from_acc, name="from accelerometer", legendgroup="acc", line_color="#1f77b4", showlegend=row == 1), row=row, col=1)
    fig.add_trace(go.Scatter(x=t, y=from_gyr, name="from gyroscope (integrated)", legendgroup="gyr", line_color="#ff7f0e", showlegend=row == 1), row=row, col=1)
fig.update_yaxes(title_text="roll [°]", row=1, col=1)
fig.update_yaxes(title_text="pitch [°]", row=2, col=1)
fig.update_xaxes(title_text="time [s]", row=2, col=1)
fig.update_layout(height=600, hovermode="x unified", title="Tilt: accelerometer vs. gyroscope")
fig.show()

### Complementary filter
Combines both: trust the gyro short-term, the accelerometer long-term. This is how phones compute orientation.

In [34]:
ALPHA = 0.98  # weight of the gyro (short-term), 1 - ALPHA for the accelerometer (long-term)

# Each step: take the previous angle + gyro change, then pull it slightly towards the accelerometer angle
roll_cf = np.zeros(len(t))
roll_cf[0] = roll_acc[0]
for k in range(1, len(t)):
    gyro_step = roll_cf[k - 1] + np.degrees(gyr[k, 0] * dt_s[k])
    roll_cf[k] = ALPHA * gyro_step + (1 - ALPHA) * roll_acc[k]

# Black line should be smooth (like the gyro) but not drift away (like the accelerometer)
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=roll_acc, name="accelerometer", opacity=0.6))
fig.add_trace(go.Scatter(x=t, y=roll_gyr, name="gyroscope"))
fig.add_trace(go.Scatter(x=t, y=roll_cf, name="complementary filter", line=dict(color="black", width=2)))
fig.update_layout(height=400, hovermode="x unified", xaxis_title="time [s]", yaxis_title="roll [°]", title="Complementary filter")
fig.show()

## 7. Gyro drift
Integrating only the still part shows how fast the angle would drift because of bias.
Small drift (< ~1°/min) = good gyro.

In [35]:
# Integrate the gyro over the still samples only: ideally the angle stays at 0
dt_still = np.median(np.diff(t))
drift = np.degrees(np.cumsum(gyr_still, axis=0) * dt_still)
t_still = np.arange(len(gyr_still)) * dt_still

fig = go.Figure()
for i, axis in enumerate("xyz"):
    fig.add_trace(go.Scatter(x=t_still, y=drift[:, i], name=axis, line_color=COLORS[axis]))
fig.update_layout(
    height=400, hovermode="x unified", xaxis_title="still time [s]", yaxis_title="integrated angle [°]", title="Gyro drift while still (should stay near 0)"
)
fig.show()

# Bias extrapolated to one minute
bias_deg_per_min = np.degrees(gyr_still.mean(axis=0)) * 60
print(f"drift rate from bias [°/min]: {bias_deg_per_min.round(2)}")

drift rate from bias [°/min]: [ 2.44 -5.03  1.85]


## Summary – what "trustworthy" looks like
| Check | Good sign |
|---|---|
| Sampling | steady interval, no gaps |
| \|a\| at rest | ≈ 9.81 m/s² (within ~0.1) |
| Accel noise at rest | ~0.01–0.05 m/s² |
| Gyro bias at rest | ≈ 0 (< 0.01 rad/s) |
| Tilt comparison | accel and gyro curves follow each other |
| Drift | a few °/min or less |